# 일괄 탐지 (여러 영상 → 교차로별 DB)

여러 영상을 차례로 **객체 탐지 + 궤적 저장**합니다. 분석라인·엑셀은 쓰지 않습니다. 로컬 GUI의 `일괄 탐지`와 같은 코드입니다.

- 영상 이름: `<교차로명>_<세션명>.mp4` (예: `경원교차로_오전첨두.mp4`)
- 저장: 교차로마다 `vm/count_car_ver6.0/output/db_snapshots/tracks_<교차로명>.sqlite`, 영상마다 세션 `<교차로명> <세션명>`
- 런타임 유형을 **GPU**로 설정한 뒤 위에서부터 실행하세요. 모두 끝나면 GPU 런타임을 자동으로 반납합니다
- **세션이 끊기면 다시 위에서부터 실행**하세요. 완료된 영상은 건너뛰고 끊긴 영상부터 다시 처리합니다(기존 데이터는 그대로 보존)
- 영상 목록은 A(Drive 폴더) 또는 B(GUI `일괄 탐지` 창의 `Colab용 내보내기` 목록 파일) 중 하나로 정합니다

In [ ]:
from google.colab import drive, runtime
drive.mount('/content/drive')

import os, sys, subprocess
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')
ROOT = DRIVE / 'vm' / 'count_car_ver6.0'
os.chdir(ROOT)
sys.path[:0] = [str(ROOT), str(ROOT / 'colab')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'colab/requirements_colab.txt', 'lap'], check=True)

# ===== 설정 =====
# A: Drive 폴더의 영상 (FILES 를 비우면 폴더 안의 <교차로명>_<세션명> 영상 전체, 이름순)
VIDEO_DIR = 'video/김해교통정비'          # MyDrive 아래 경로
FILES = []                                # 예: ['경원교차로_오전첨두.mp4', '봉황교사거리_오전첨두.mp4']
# B: GUI 일괄 탐지 창 'Colab용 내보내기' 목록 파일. 지정하면 A 대신 이것을 쓴다(영상별 모델 포함)
JOBS_FILE = ''                            # 예: 'colab/batch_jobs/batch_20261008_1700.json'

MODEL = ''          # 비우면 기본 models/yolo8m_v1.pt. 예: 'models/yolo26n_v1.pt' (B 의 영상별 모델이 우선)
IMGSZ = ''          # 비우면 기본 1920. 'auto'(모델 학습 크기) 또는 숫자
PRECISION = 'fp16'  # 'fp16'(빠름, 결과 사실상 같음) 또는 'fp32'
EXISTING = 'skip'   # 완료 기록 없이 이미 있는 세션: 'skip'(건너뜀) 또는 'overwrite'(다시 처리)
AUTO_UNASSIGN = True  # 모두 끝나면 GPU 런타임 반납
# ================

In [ ]:
import importlib, batch_snippet
importlib.reload(batch_snippet)

jobs = batch_snippet.plan_jobs(
    video_dir=None if JOBS_FILE else DRIVE / VIDEO_DIR,
    files=FILES,
    jobs_file=ROOT / JOBS_FILE if JOBS_FILE else None,
)
base = {'precision': PRECISION}
if MODEL:
    base['model_path'] = MODEL
if IMGSZ:
    base['yolo_imgsz'] = 'auto' if IMGSZ == 'auto' else int(IMGSZ)
batch_snippet.print_plan(jobs, ROOT / 'output' / 'db_snapshots')

## (선택) FP16 확인 — 처음 한 번만

첫 영상 앞 3분으로 FP32와 FP16의 속도와 궤적 수를 비교합니다(임시 DB, 실제 DB는 건드리지 않음). 궤적 수 차이가 1–2% 안이고 FP16이 빠르면 `PRECISION = 'fp16'` 그대로 쓰면 됩니다. 건너뛰어도 됩니다.

In [ ]:
batch_snippet.fp16_check(jobs[0].video, root=ROOT, minutes=3, base_overrides=base)

## 실행

진행 상황이 아래에 표시됩니다. 영상 하나가 끝날 때마다, 긴 영상은 15분마다 Drive에 저장합니다.

In [ ]:
import time, torch
assert torch.cuda.is_available(), 'GPU 가 없습니다. 런타임 유형을 GPU 로 바꾸고 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0), '| 시작:', time.strftime('%Y-%m-%d %H:%M:%S'))
try:
    results = batch_snippet.run_colab_batch(
        jobs, root=ROOT, base_overrides=base, existing=EXISTING,
        remount=lambda: drive.mount('/content/drive', force_remount=True),
    )
    batch_snippet.print_summary(results)
finally:
    print('종료:', time.strftime('%Y-%m-%d %H:%M:%S'))
    if AUTO_UNASSIGN:
        runtime.unassign()